Formatação inicial de Dataframe Unindo os DataSets 

In [19]:
import pandas as pd
import numpy as np
import glob
import os

def recuperar_valor_real(valor):

    s = str(valor).strip()
    # 1. Se for uma data (05/02/2021) ou contiver '/', o Sheets corrompeu.
    # Vamos extrair o Dia e o Mês e remontar o decimal (Ex: 05/02 vira 5.2)
    if '/' in s:
        partes = s.split('/')
        try:
            return float(f"{partes[0]}.{partes[1]}")
        except:
            return np.nan

    # 2. Limpeza de formatação brasileira
    s = s.replace('.', '')  # Remove ponto de milhar (se houver)
    s = s.replace(',', '.')  # Troca vírgula decimal por ponto
    
    try:
        n = float(s)
        # Filtro de Sanidade: Se o valor for um flag do INMET ou absurdo
        if n <= -999 or n > 5000: 
            return np.nan
        return n
    except:
        return np.nan

caminho_pasta = 'dados_inmet'
arquivos = glob.glob(os.path.join(caminho_pasta, "*.CSV"))

lista_final = []

print(f"--- Iniciando Unificação de {len(arquivos)} arquivos ---\n")

for f in arquivos:
    try:
        # 1. Lendo o arquivo bruto como STRING (para evitar conversão errada em data)
        df_temp = pd.read_csv(f, sep=';', encoding='latin-1', skiprows=8, dtype=str)
        df_temp.columns = df_temp.columns.str.strip().str.upper()
        
        # 2. Localização Dinâmica de Colunas
        def encontrar_coluna(termos):
            for termo in termos:
                for col in df_temp.columns:
                    if termo in col: return col
            return None

        mapa = {
            'DATA': encontrar_coluna(['DATA']),
            'HORA': encontrar_coluna(['HORA']),
            'CHUVA': encontrar_coluna(['PRECIPITAÇÃO', 'CHUVA']),
            'PRESSAO': encontrar_coluna(['PRESSAO ATMOSFERICA']),
            'TEMP': encontrar_coluna(['TEMPERATURA DO AR - BULBO SECO', 'TEMP_AR']),
            'UMID': encontrar_coluna(['UMIDADE RELATIVA']),
            'VENTO': encontrar_coluna(['RAJADA', 'VENTO_RAJADA']),
            'RAD': encontrar_coluna(['RADIACAO'])
        }

        # 3. Filtragem e Renomeação (AQUI filtramos antes de limpar)
        mapa_limpo = {v: k for k, v in mapa.items() if v is not None}
        df_filtrado = df_temp[list(mapa_limpo.keys())].rename(columns=mapa_limpo)
        
        # 4. APLICAÇÃO DA LIMPEZA (Agora usamos os nomes novos: TEMP, CHUVA, etc.)
        # Isso resolve o problema das "datas" que o Sheets criou
        colunas_para_limpar = ['CHUVA', 'PRESSAO', 'TEMP', 'UMID', 'VENTO', 'RAD']
        for col in colunas_para_limpar:
            if col in df_filtrado.columns:
                df_filtrado[col] = df_filtrado[col].apply(recuperar_valor_real)

        # 5. Limpeza de Linhas Inúteis
        df_filtrado = df_filtrado.dropna(subset=['DATA', 'HORA'])
        
        # Definimos que a linha precisa de valores mínimos para ser útil
        df_filtrado = df_filtrado.dropna(thresh=4)
        
        # Remove se os principais sensores estiverem nulos (evita linhas vazias de sensores offline)
        df_filtrado = df_filtrado.dropna(subset=['CHUVA', 'TEMP'], how='all')

        # ADICIONAMOS APENAS O FILTRADO NA LISTA
        lista_final.append(df_filtrado)
        
        print(f"✅ {os.path.basename(f)}: {len(df_filtrado)} linhas processadas.")

    except Exception as e:
        print(f"❌ Erro no arquivo {f}: {e}")

    # 3. CRIAÇÃO DO DF_MASTER (Unificando a lista)
if lista_final:
    df_master = pd.concat(lista_final, axis=0, ignore_index=True)
    print("\n🚀 df_master criado com sucesso!")
else:
    print("\n⚠️ Alerta: lista_final está vazia. Verifique os arquivos.")    

--- Iniciando Unificação de 10 arquivos ---

✅ INMET_NE_CE_A305_FORTALEZA_01-01-2017_A_31-12-2017.CSV: 8747 linhas processadas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2018_A_31-12-2018.CSV: 8622 linhas processadas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2019_A_31-12-2019.CSV: 6356 linhas processadas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2020_A_31-12-2020.CSV: 8777 linhas processadas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2021_A_31-12-2021.CSV: 8179 linhas processadas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2022_A_31-12-2022.CSV: 8712 linhas processadas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2023_A_31-12-2023.CSV: 8724 linhas processadas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2024_A_31-12-2024.CSV: 2911 linhas processadas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2025_A_30-11-2025.CSV: 162 linhas processadas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2025_A_31-12-2025.CSV: 162 linhas processadas.

🚀 df_master criado com sucesso!


Conversão para horário local e formatação de data e hora

In [20]:
#convertendo string para int
df_master['HORA_NUM'] = df_master['HORA'].str.extract(r'(\d+)').astype(int)

#A formatação das horas de 2019 a 2025 são diferentes dos outros anos,
#precisamos fazer a padronização
df_master['HORA_NUM'] = df_master['HORA_NUM'].apply(lambda x: x // 100 if x >= 100 else x)

#Agora precisamos padronizar a coluna 'HORA' para o formato Brasileiro horário local

#retirando do formato UTC e unindo as colunas 'DATA' e 'HORA'
df_master['dt_utc'] = pd.to_datetime(
    df_master['DATA'].str.replace('/', '-') + ' ' +
    df_master['HORA_NUM'].astype(str).str.zfill(2) + ':00'
                                )
#Convertendo para o fuso horário de fortaleza (UTC-3)
df_master['dt_local'] = df_master['dt_utc'] - pd.Timedelta(hours=3)

#Extraindo as informações locais para treinar o modelo
df_master['HORA_LOCAL'] = df_master['dt_local'].dt.hour
df_master['MES_REFERENCIA'] = df_master['dt_local'].dt.date

#Descartando as colunas antigas 'DATA', 'HORA' e as auxiliates 'dt_utc', 'HORA_NUM'
df_master = df_master.drop(columns = ['DATA', 'HORA', 'dt_utc', 'HORA_NUM'])


Formalização de linhas com dados vazios

In [21]:
import numpy as np

# Converter o erro do INMET (-9999) em Valor Nulo Real (NaN)
colunas_clima = ['CHUVA', 'PRESSAO', 'TEMP', 'UMID', 'VENTO', 'RAD']

df_master[colunas_clima] = df_master[colunas_clima].replace(-9999, np.nan)

# 2. Preenchimento Inteligente: Noite = Radiação 0
# Considerando que em Fortaleza o sol se põe por volta das 18h e nasce às 05:30h
condicao_noite = (df_master['HORA_LOCAL'] >= 18) | (df_master['HORA_LOCAL'] <= 5)

df_master.loc[condicao_noite, 'RAD'] = df_master.loc[condicao_noite, 'RAD'].fillna(0)

# 3. Agora sim, aplica a interpolação linear apenas para buracos pequenos (máx  imo 2h)
df_master['RAD'] = df_master['RAD'].interpolate(method='linear', limit=2)

# 4. Se ainda sobrarem nulos (buracos grandes), preenchemos com 0 para não quebrar o modelo
# (Ou você pode optar por dropna() se quiser apenas dados perfeitos)
df_master['RAD'] = df_master['RAD'].fillna(0)

Tratamento de Nulos

In [22]:
df_copy = df_master.copy()

# Remover duplicados
df_copy = df_copy.drop_duplicates(subset='dt_local', keep='last')

# Garantir que o DATA_LOCAL É DATATIME
df_copy['MES_REFERENCIA'] = pd.to_datetime(df_copy['MES_REFERENCIA'])

# Ordenar 
df_copy = df_copy.sort_values('dt_local')

# Colunas
colunas = ['CHUVA','PRESSAO','TEMP','UMID','VENTO','RAD']

# Interpolação
for col in colunas:
    df_copy[col] = df_copy[col].interpolate(limit=6)

# Identificar dias ruins
dias_remover = set()

for col in colunas:
    is_nan = df_copy[col].isna()
    grupos = (is_nan != is_nan.shift()).cumsum()
    
    blocos = df_copy[is_nan].groupby(grupos).size()
    blocos_grandes = blocos[blocos > 6].index
    
    dias = df_copy.loc[grupos.isin(blocos_grandes), 'MES_REFERENCIA']
    dias_remover.update(pd.to_datetime(dias).dt.date)

# Remover dias ruins
df_copy = df_copy[~df_copy['MES_REFERENCIA'].dt.date.isin(dias_remover)]

# Remover NaN restantes
df_copy = df_copy.dropna()

# Verificar
df_copy.isna().sum()



CHUVA             0
PRESSAO           0
TEMP              0
UMID              0
VENTO             0
RAD               0
dt_local          0
HORA_LOCAL        0
MES_REFERENCIA    0
dtype: int64

Criação do Delta

In [23]:
# Ordenar 
df_copy = df_copy.sort_values('dt_local')

# Cria calculo para o DELTA_P e e limita 2 casas decimais
df_copy['DELTA_P'] = (df_copy['PRESSAO'] - df_copy['PRESSAO'].shift(3)).round(2)

# Remover todas as linhas onde a coluna DELTA_P é NaN e reseta o índice
df_copy = df_copy.dropna(subset=['DELTA_P']).reset_index(drop=True)

# Deixar maiúsculo
df_copy = df_copy.rename(columns={'dt_local': 'DT_LOCAL'})

# Mostrar tabela final
df_copy

,CHUVA,PRESSAO,TEMP,UMID,VENTO,RAD,DT_LOCAL,HORA_LOCAL,MES_REFERENCIA,DELTA_P
0,0.0,1008.7,27.4,76.0,7.0,0.0,2017-01-01 00:00:00,0,2017-01-01,-0.3
1,0.0,1007.8,27.1,76.0,7.0,0.0,2017-01-01 01:00:00,1,2017-01-01,-1.3
2,0.0,1007.5,27.0,75.0,6.6,0.0,2017-01-01 02:00:00,2,2017-01-01,-1.6
3,0.0,1007.2,26.8,76.0,6.6,0.0,2017-01-01 03:00:00,3,2017-01-01,-1.5
4,0.0,1007.1,26.7,77.0,5.3,0.0,2017-01-01 04:00:00,4,2017-01-01,-0.7
...,...,...,...,...,...,...,...,...,...,...
59754,0.0,1009.7,31.2,60.0,7.0,1.3,2025-01-30 11:00:00,11,2025-01-30,2.6
59755,0.0,1009.3,31.8,55.0,7.3,1.3,2025-01-30 12:00:00,12,2025-01-30,-0.4
59756,0.0,1008.1,31.8,56.0,7.5,0.0,2025-01-30 13:00:00,13,2025-01-30,-1.7
59757,0.0,1006.7,31.3,57.0,7.1,0.0,2025-01-30 14:00:00,14,2025-01-30,-3.0


Soma de dados climaticos obtidos por mês 

In [24]:

# Garantir que a coluna de data é do tipo datetime
df_copy['MES_REFERENCIA'] = pd.to_datetime(df_copy['MES_REFERENCIA'])

# Criar o dicionário de agregação
# Isso permite tratar cada coluna de um jeito diferente no mesmo comando
regras = {
    'CHUVA': 'sum',
    'TEMP': 'mean',
    'UMID': 'mean',
    'PRESSAO': 'mean'
}

# Agrupar por Mês (MS = Month Start) e aplicar as regras
df_mensal = df_copy.resample('MS', on='MES_REFERENCIA').agg(regras).round(2).reset_index()

#Organizar o nome das colunas para o seu "Dataset Mestre"
df_mensal.columns = ['MES_REFERENCIA', 'Chuva_Acumulada', 'Temp_Media_Mensal', 'Umid_Media', 'Pres_Media']

Interpolação de dados de meses faltantes

In [25]:
# Verifica quais meses ficaram com valores nulos (NaN) após a limpeza e agrupamento
meses_vazios = df_mensal[df_mensal['Temp_Media_Mensal'].isnull()]
print("Meses sem dados do INMET:")
print(meses_vazios['MES_REFERENCIA'])

Meses sem dados do INMET:
26   2019-03-01
27   2019-04-01
67   2022-08-01
92   2024-09-01
93   2024-10-01
Name: MES_REFERENCIA, dtype: datetime64[s]


In [26]:
# Preenche os meses vazios estimando os valores entre o mês anterior e o próximo
df_mensal[['Temp_Media_Mensal', 'Umid_Media', 'Pres_Media']] = df_mensal[['Temp_Media_Mensal', 'Umid_Media', 'Pres_Media']].interpolate(method='linear')

# Verificar os meses que foram preenchidos
meses_preenchidos = ['2019-03-01', '2019-04-01', '2022-08-01', '2024-09-01', '2024-10-01']
for x in meses_preenchidos:
    if x in df_mensal['MES_REFERENCIA'].astype(str).values:
        print(f"✅ Mês {x} preenchido com o seguinte valor: {df_mensal[df_mensal['MES_REFERENCIA'] == x][['Temp_Media_Mensal', 'Umid_Media', 'Pres_Media']].values[0]}")

# Arredondar os valores para 2 casas decimais
df_mensal[['Temp_Media_Mensal', 'Umid_Media', 'Pres_Media']] = df_mensal[['Temp_Media_Mensal', 'Umid_Media', 'Pres_Media']].round(2)
# Mostrar o resultado final
print("\n--- Dataset Mensal Final ---")
print(df_mensal)

✅ Mês 2019-03-01 preenchido com o seguinte valor: [  28.44666667   72.69666667 1008.71333333]
✅ Mês 2019-04-01 preenchido com o seguinte valor: [  27.74333333   76.02333333 1008.90666667]
✅ Mês 2022-08-01 preenchido com o seguinte valor: [  26.7    69.88 1010.52]
✅ Mês 2024-09-01 preenchido com o seguinte valor: [  30.64666667   50.30333333 1010.44333333]
✅ Mês 2024-10-01 preenchido com o seguinte valor: [  30.94333333   50.60666667 1009.68666667]

--- Dataset Mensal Final ---
   MES_REFERENCIA  Chuva_Acumulada  Temp_Media_Mensal  Umid_Media  Pres_Media
0      2017-01-01            105.2              27.65       72.81     1008.44
1      2017-02-01            209.6              27.30       75.68     1008.18
2      2017-03-01            475.0              26.46       82.21     1008.32
3      2017-04-01            334.2              27.07       79.31     1008.42
4      2017-05-01            129.8              27.29       76.54     1009.36
..            ...              ...                

Exportando Dados

In [27]:
#Instala o dataset com os dados formalizados em formato csv
df_mensal.to_csv('dados_formatados/FORTALEZA_DADOS_CLIMATICOS.csv', sep=';', decimal=',', index=False, encoding='utf-8-sig')
print(f"\nSUCESSO! Total de {len(df_mensal)} linhas consolidadas em 'FORTALEZA_DADOS_CLIMATICOS'")


SUCESSO! Total de 97 linhas consolidadas em 'FORTALEZA_DADOS_CLIMATICOS'
